In [ ]:
# FLORES-200 翻译评估

本地 vLLM + FLORES-200；入口 **`run_evaluation.py`** + `evaluation_config.json`。  
模块在 **`src/lowres_translation/`**；数据：`download_flores.py`、`download_ccmatrix.py`。

---

## 1. 一键运行（自动启动 vLLM 再评估）

```bash
# 默认 batch 预设 zh_en_cross_lowres，每语对 50 条试跑
python3 run_evaluation.py --limit 50

# 单语对（英→中）
python3 run_evaluation.py --no-serve --mode single --config eng_Latn-zho_Hans --limit 100
```

---

## 2. 分步运行（先起 vLLM，再跑评估）

**终端 1 — 启动 vLLM（端口 8005）：**
```bash
source .venv/bin/activate
NVIDIA_VISIBLE_DEVICES=0,1,2,3 vllm serve ./huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a/  --port 8005   --tensor-parallel-size 4  --served-model-name Qwen/Qwen3.5-9B

NVIDIA_VISIBLE_DEVICES=0,1,2,3 vllm serve ./huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a/ --port 8005 --tensor-parallel-size 1 --max-model-len 262144 --language-model-only --served-model-name Qwen/Qwen3.5-9B

```

**终端 2 — 只跑评估（加 `--no-serve`）：**
```bash
source .venv/bin/activate
python3 run_evaluation.py --no-serve --limit 50
# 或单语对
python3 run_evaluation.py --no-serve --mode single --config eng_Latn-zho_Hans
```

---

## 3. 模块内 CLI（开发用）

API 默认：`http://localhost:8005/v1`

```bash
PYTHONPATH=src python3 -m lowres_translation.eval_single --config eng_Latn-zho_Hans --limit 50
```

批量评估请用根目录 **`run_evaluation.py`** + 配置文件。

---

## 4. 可选参数（run_evaluation.py）

详见 **README.md**。常用：

| 参数 | 说明 |
|------|------|
| `--config-file` | 评估 JSON 配置路径 |
| `--no-serve` | 不启动 vLLM，只跑评估 |
| `--gpus 4,5` | 指定 GPU（不填会交互询问） |
| `--port` / `--base-url` | API 地址 |
| `--model` | 模型名或本地路径 |
| `--limit` | 每语对评估条数 |
| `--metrics` | `bleu` 或 `bleu,comet` |